In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pathlib import Path


In [ ]:
PROJECT_ROOT = Path("/workspaces/spark")


In [ ]:
spark = (SparkSession.builder
         .appName("shopflow-silver-orders")
         .master("local[*]")
         .getOrCreate())

In [ ]:
spark.sparkContext.setJobDescription("READ bronze orders")
bronze = spark.read.parquet(str(PROJECT_ROOT / "data/bronze/orders"))
bronze.show(5, truncate=False)

lista ei product_id validi, per il controllo di integrità referenziale


In [ ]:
valid_products = (
    spark.read.parquet(str(PROJECT_ROOT / "data/bronze/products"))
    .select("product_id")
)

valid_products.show(5, truncate=False)

list comprehension Python standard (non specifica di Spark): itera su ogni Row della lista appena ottenuta da .collect(), estrae solo il valore di product_id da ciascuna, e costruisce una lista Python semplice

In [ ]:
valid_product_ids = [row.product_id for row in valid_products.collect()]
print(valid_product_ids[:5])

In [ ]:
# 1. flag di validazione, indipendenti tra loro

checked = (
    bronze
    .withColumn("err_null_customer", F.col("customer_id").isNull())
    .withColumn("err_negative_qty", F.col("quantity") < 0)
    .withColumn("err_invalid_product", ~F.col("product_id").isin(valid_product_ids))
)

checked.show(5, truncate=False)

2. deduplicazione via window function

In [ ]:
w = Window.partitionBy("order_id").orderBy(F.col("order_ts").asc())
ranked = checked.withColumn("rn", F.row_number().over(w))

ranked.show(5, truncate=False)

Per vedere solo i duplicati (quelli con rn > 1

In [ ]:
ranked.filter(F.col("rn") > 1).select("order_id", "order_ts", "rn").show(20, truncate=False)

definiamo condizioni record con errori (has error) e record duplicati (is duplicated) 

In [ ]:
has_error = F.col("err_null_customer") | F.col("err_negative_qty") | F.col("err_invalid_product")
is_duplicate = F.col("rn") > 1

3. split record: errori, duplicati (scartati silenziosamente), puliti + creazione df


In [ ]:
duplicate = ranked.filter(is_duplicate)
errors = ranked.filter(has_error)
clean = ranked.filter(~has_error & ~is_duplicate)

print("duplicate")
duplicate.show(5, truncate=False)
print(f"Count: {duplicate.count()}")

print("\n\n")

print("errors")
errors.show(5, truncate=False)
print(f"Count: {errors.count()}")

print("\n\n")

print("clean")
clean.show(5, truncate=False)
print(f"Count: {clean.count()}")


In [ ]:
spark.sparkContext.setJobDescription("WRITE silver orders")
(
    clean
    .withColumn("order_date", F.to_date("order_ts"))
    .drop("rn", "err_null_customer", "err_negative_qty", "err_invalid_product")
    .coalesce(4)
    .write
    .mode("overwrite")
    .partitionBy("order_date")
    .parquet(str(PROJECT_ROOT / "data/silver/orders"))
)


In [ ]:
spark.sparkContext.setJobDescription("WRITE errors orders")
(
    errors
    .drop("rn")
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(PROJECT_ROOT / "data/silver/errors_orders"))
)

In [ ]:
spark.stop()